In [1]:
#!pip install pymupdf
#!pip install Pillow

In [2]:
import logging
import boto3
from pdf_vectorizer_EC2 import (
    batch_process_pdfs,
    clear_vector_index,
    list_s3_folders,
    verify_vector_index_empty,
)

# Logging

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# S3 configuration

In [4]:
SOURCE_BUCKET = "bhitech-minelogx-poc-legislation-documents"
VECTOR_BUCKET = "bhitech-minelogx-poc-legislation-documents-vector"
INDEX_NAME = "bhitech-minelogx-chunkspdfs"
PDF_PREFIX = "docs/"  # root folder to scan
FOLDER_LIMIT = None  # set to an int (e.g. 10) to process a subset only

# AWS configuration

In [5]:
AWS_REGION = "us-east-1"

# PDF splitting configuration

In [6]:
MAX_PAGES_PER_CHUNK = 2 # pages per PDF page-chunk sent to the extractor
CHUNK_SIZE_LIMIT_MB = 4.5    # compress the chunk if it exceeds this size
COMPRESSION_TARGET_MB = 4.0    # target size after compression
COMPRESSION_QUALITY_STEPS = [85, 70, 50, 30]  # JPEG quality levels tried in order

# Text extraction configuration

In [7]:
# fitz (PyMuPDF) is used for all extraction — no external API calls.
# preserve_layout=True keeps column/block reading order (useful for
# multi-column regulatory documents). False is faster and cleaner for
# single-column PDFs.

PRESERVE_LAYOUT = False

# Text chunking configuration

In [8]:
# chunk_strategy options: "length" | "section"
# Use "section" for well-structured documents with clear headings.
# Use "length" for dense or unstructured text (scanned PDFs, raw exports).
 
CHUNK_STRATEGY = "length"
MAX_EMBED_CHARS = 300   # maximum characters per embed sub-chunk
CHUNK_OVERLAP = 100     # characters repeated between consecutive length-based chunks
SECTION_HEADING_PATTERN = r"(?m)^#{1,6}\s+.+$|^[A-Z][^\n]{0,80}\n[-=]{3,}"  # Markdown + underline headings

# Embedding configuration

In [9]:
EMBEDDINGS_ENDPOINT = "http://ec2-3-208-23-94.compute-1.amazonaws.com:11434"
EMBEDDINGS_MODEL    = "mxbai-embed-large"
EMBEDDINGS_TIMEOUT  = 30    # seconds per embedding request

# Storage configuration

In [10]:
STORE_VECTORS       = True   # set to False for a dry run (no writes to S3 Vectors)
VECTOR_BATCH_SIZE   = 100    # vectors per put_vectors call

# Pipeline

In [11]:
logger.info("Starting PDF vectorization pipeline")
logger.info("Source:  s3://%s/%s", SOURCE_BUCKET, PDF_PREFIX)
logger.info("Target:  %s / %s", VECTOR_BUCKET, INDEX_NAME)
logger.info(
    "Extractor: fitz (preserve_layout=%s)  |  Chunk strategy: %s  |  Embedding model: %s",
    PRESERVE_LAYOUT, CHUNK_STRATEGY, EMBEDDINGS_MODEL,
)

# Create AWS clients once and reuse across the entire run
# No Bedrock or Textract clients needed — embedding runs via Ollama HTTP
s3_client        = boto3.client("s3",        region_name=AWS_REGION)
s3vectors_client = boto3.client("s3vectors", region_name=AWS_REGION)


2026-06-03 21:41:56  INFO      __main__  Starting PDF vectorization pipeline
2026-06-03 21:41:56  INFO      __main__  Source:  s3://bhitech-minelogx-poc-legislation-documents/docs/
2026-06-03 21:41:56  INFO      __main__  Target:  bhitech-minelogx-poc-legislation-documents-vector / bhitech-minelogx-chunkspdfs
2026-06-03 21:41:56  INFO      __main__  Extractor: fitz (preserve_layout=False)  |  Chunk strategy: length  |  Embedding model: mxbai-embed-large
2026-06-03 21:41:56  INFO      botocore.credentials  Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


In [12]:
# Step 1: Discover folders
folders = [PDF_PREFIX + "/"] if not PDF_PREFIX.endswith("/") else [PDF_PREFIX]
logger.info("Using folder directly: %s", folders)

2026-06-03 21:41:56  INFO      __main__  Using folder directly: ['docs/']


In [13]:
 # Step 2: Clear existing index (skip on dry runs)
if STORE_VECTORS:
    logger.info("Clearing index '%s' before re-indexing...", INDEX_NAME)
    deleted = clear_vector_index(
        vector_bucket_name=VECTOR_BUCKET,
        index_name=INDEX_NAME,
        s3vectors_client=s3vectors_client,
        region=AWS_REGION,
    )
    logger.info("Deleted %d existing vectors.", deleted)

    if not verify_vector_index_empty(
        vector_bucket_name=VECTOR_BUCKET,
        index_name=INDEX_NAME,
        s3vectors_client=s3vectors_client,
        region=AWS_REGION,
    ):
        logger.error("Index is not empty after clearing. Aborting to avoid duplicates.")
else:
    logger.info("STORE_VECTORS=False — skipping index clear (dry run).")

2026-06-03 21:41:56  INFO      __main__  Clearing index 'bhitech-minelogx-chunkspdfs' before re-indexing...
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  Clearing index: bhitech-minelogx-poc-legislation-documents-vector/bhitech-minelogx-chunkspdfs
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  Index confirmed empty. Total deleted: 0
2026-06-03 21:41:56  INFO      __main__  Deleted 0 existing vectors.
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  Index confirmed empty.


In [ ]:
# Step 3: Run batch pipeline
all_vectors = batch_process_pdfs(
    bucket_name=SOURCE_BUCKET,
    folders=folders,
    vector_bucket_name=VECTOR_BUCKET,
    index_name=INDEX_NAME,
    s3_client=s3_client,
    s3vectors_client=s3vectors_client,
    region=AWS_REGION,
    max_pages_per_chunk=MAX_PAGES_PER_CHUNK,
    chunk_size_limit_mb=CHUNK_SIZE_LIMIT_MB,
    compression_target_mb=COMPRESSION_TARGET_MB,
    compression_quality_steps=COMPRESSION_QUALITY_STEPS,
    preserve_layout=PRESERVE_LAYOUT,
    chunk_strategy=CHUNK_STRATEGY,
    max_embed_chars=MAX_EMBED_CHARS,
    chunk_overlap=CHUNK_OVERLAP,
    section_heading_pattern=SECTION_HEADING_PATTERN,
    embeddings_endpoint=EMBEDDINGS_ENDPOINT,
    embeddings_model=EMBEDDINGS_MODEL,
    embeddings_timeout=EMBEDDINGS_TIMEOUT,
    store_vectors=STORE_VECTORS,
    vector_batch_size=VECTOR_BATCH_SIZE,
)

logger.info("Pipeline complete. Total vectors produced: %d", len(all_vectors))

2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  ### BATCH PROCESSING | source: s3://bhitech-minelogx-poc-legislation-documents | target: bhitech-minelogx-poc-legislation-documents-vector/bhitech-minelogx-chunkspdfs | strategy: length | model: mxbai-embed-large ###
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  Processing: docs/CELEX_32014L0034_EN_TXT.pdf
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  === Starting: docs/CELEX_32014L0034_EN_TXT.pdf (strategy=length) ===
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  [1/3] Downloading PDF from s3://bhitech-minelogx-poc-legislation-documents/docs/CELEX_32014L0034_EN_TXT.pdf
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2        Size: 1.01 MB
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  [2/3] Splitting into chunks (max 2 pages)...
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  PDF: 48 pages -> 24 chunks (2 pages/chunk)
2026-06-03 21:41:56  INFO      pdf_vectorizer_EC2  [3/3] Processing 24 chunks (fitz -> mxbai-emb

In [ ]:
import requests

response = requests.get(
    "http://ec2-3-208-23-94.compute-1.amazonaws.com:11434/api/tags",
    timeout=10
)
print(response.json())

In [ ]:
# ============================================================
# PIPELINE DIAGNOSTICS — tests each stage independently
# Run this to find exactly where the pipeline is failing
# ============================================================

import boto3
import fitz
import requests
import io

_s3  = boto3.client("s3",        region_name=AWS_REGION)
_s3v = boto3.client("s3vectors", region_name=AWS_REGION)

PASS = "✅"
FAIL = "❌"
WARN = "⚠️"

print("=" * 60)
print("PIPELINE DIAGNOSTICS")
print("=" * 60)

# ----------------------------------------------------------
# Stage 1: S3 — can we list and download a PDF?
# ----------------------------------------------------------
print("\n[1/4] S3 ACCESS")

try:
    # Normalize prefix: ensure it ends with "/" so we list files inside the folder
    prefix = PDF_PREFIX if PDF_PREFIX.endswith("/") else PDF_PREFIX + "/"

    response = _s3.list_objects_v2(Bucket=SOURCE_BUCKET, Prefix=prefix)
    contents = response.get("Contents", [])

    # Filter to PDF files sitting directly in this folder (no sub-folder)
    pdf_keys = [
        o["Key"] for o in contents
        if o["Key"].lower().endswith(".pdf")
        and o["Key"] != prefix
        and "/" not in o["Key"][len(prefix):]   # exclude files in sub-folders
    ]

    print(f"  Found {len(pdf_keys)} PDF(s) directly under s3://{SOURCE_BUCKET}/{prefix}")
    for k in pdf_keys:
        print(f"         {k}")

    if not pdf_keys:
        print(f"  {WARN}  No PDFs found — check that PDF_PREFIX is correct")
        test_pdf_bytes = None
    else:
        # Download the first PDF for the remaining stages
        test_key       = pdf_keys[0]
        obj            = _s3.get_object(Bucket=SOURCE_BUCKET, Key=test_key)
        test_pdf_bytes = obj["Body"].read()
        size_mb        = len(test_pdf_bytes) / (1024 * 1024)
        print(f"  {PASS}  Downloaded for testing: {test_key} ({size_mb:.2f} MB)")

except Exception as e:
    print(f"  {FAIL}  S3 error: {e}")
    test_pdf_bytes = None

# ----------------------------------------------------------
# Stage 2: fitz — can we extract text from the PDF?
# ----------------------------------------------------------
print("\n[2/4] TEXT EXTRACTION (fitz)")

test_text = None

if test_pdf_bytes:
    try:
        pdf_doc    = fitz.open(stream=test_pdf_bytes, filetype="pdf")
        total_pages = len(pdf_doc)
        # Extract text from the first page only as a quick test
        test_text  = pdf_doc[0].get_text("text").strip()
        pdf_doc.close()

        if test_text:
            print(f"  {PASS}  Extracted text from {total_pages} pages")
            print(f"         First 200 chars: {repr(test_text[:200])}")
        else:
            print(f"  {WARN}  PDF opened but page 1 returned no text (may be scanned/image-only)")

    except Exception as e:
        print(f"  {FAIL}  fitz error: {e}")
else:
    print(f"  {WARN}  Skipped — no PDF available from Stage 1")

# ----------------------------------------------------------
# Stage 3: Ollama — can we get an embedding?
# ----------------------------------------------------------
print("\n[3/4] EMBEDDING (mxbai-embed-large)")

test_embedding = None
sample_text    = (test_text[:200] if test_text else "test mining regulation dust limit") 

try:
    resp = requests.post(
        f"{EMBEDDINGS_ENDPOINT}/api/embeddings",
        json={"model": EMBEDDINGS_MODEL, "prompt": sample_text},
        timeout=EMBEDDINGS_TIMEOUT,
    )
    resp.raise_for_status()
    test_embedding = resp.json().get("embedding", [])

    if test_embedding:
        print(f"  {PASS}  Embedding returned: {len(test_embedding)} dimensions")
        print(f"         First 5 values: {test_embedding[:5]}")
    else:
        print(f"  {FAIL}  Ollama responded 200 but embedding is empty")

except requests.exceptions.HTTPError as e:
    print(f"  {FAIL}  HTTP {e.response.status_code}: {e}")
    print(f"         Text sent ({len(sample_text)} chars): {repr(sample_text[:100])}")
except Exception as e:
    print(f"  {FAIL}  Embedding error: {e}")

# ----------------------------------------------------------
# Stage 4: S3 Vectors — can we write and read a vector?
# ----------------------------------------------------------
print("\n[4/4] S3 VECTORS (put + list)")

if test_embedding:
    try:
        # Write one test vector
        _s3v.put_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=INDEX_NAME,
            vectors=[{
                "key":      "diagnostics-test-vector",
                "data":     {"float32": test_embedding},
                "metadata": {"source": "diagnostics", "note": "safe to delete"},
            }],
        )
        print(f"  {PASS}  put_vectors succeeded")

        # Read it back
        check = _s3v.list_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=INDEX_NAME,
            maxResults=5,
        )
        found = [v["key"] for v in check.get("vectors", [])]
        print(f"  {PASS}  list_vectors returned {len(found)} vector(s): {found}")

        # Clean up the test vector
        _s3v.delete_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=INDEX_NAME,
            keys=["diagnostics-test-vector"],
        )
        print(f"  {PASS}  Test vector cleaned up")

    except Exception as e:
        print(f"  {FAIL}  S3 Vectors error: {e}")
else:
    print(f"  {WARN}  Skipped — no embedding available from Stage 3")

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("If all 4 stages show ✅, the pipeline should work end-to-end.")
print("The first ❌ is where to focus.")
print("=" * 60)